# (Opsiyonal) Olist reviews' - Translations...

* 🇧🇷 Brezilya Portekizcesi bilmiyorsanız, hadi `order_reviews` metinlerini 🇬🇧 İngilizce’ye çevirelim!

* Bunun için `pip install googletrans==4.0.2` yüklemeniz gerekecek.

☢️ Bu API ile herhangi bir sorun yaşarsanız, bunu düzeltmek için zaman harcamayın, bu package oldukça dengesiz… Aklınızda bulunsun:
- bu optional bir challenge
- Brezilya Portekizcesi ile yazılmış review’ların anlamını görmek için bazı yorumları Google Translate’e kopyalayıp yapıştırarak yine de eski yöntemle yapabilirsiniz.

## Review Translator

👉 `reviews` dataset’ini load edin ve 1-yıldız rating’e sahip review’lardan bir örnek (sample) seçin.

In [7]:
# Verileri yükle
from olist.data import Olist
data = Olist().get_data()

In [8]:
print(data.keys())

dict_keys(['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'product_category_name_translation'])


👀 20 adet düşük puan alan yorumdan oluşan bir örneklem seçin (rastgele) ve bunu bir listeye dönüştürün.

In [9]:
reviews_df = data['order_reviews']

bad_reviews = reviews_df[(reviews_df['review_score'] == 1)].dropna(subset=['review_comment_message'])

sample_reviews = bad_reviews['review_comment_message'].sample(n=20, random_state=42).tolist()

print(f"Seçilen örneklem sayısı: {len(sample_reviews)}")

Seçilen örneklem sayısı: 20


🗣 Bu göreve başlamadan önce önceden yüklediğiniz `google_translator` paketini kullanarak bu yorumları çevirin:

In [10]:
from googletrans import Translator
import pandas as pd

translator = Translator()
translation_results = []

print("Çeviri işlemi başladı, lütfen bekleyin...\n")

for text in sample_reviews:
    try:
        translation = translator.translate(text, src='pt', dest='en')
        translation_results.append({
            'original_text': text,
            'translated_text': translation.text
        })
    except Exception as e:
        translation_results.append({
            'original_text': text,
            'translated_text': f"[HATA - ÇEVRİLEMEDİ] Detay: {e}"
        })

df_translations = pd.DataFrame(translation_results)

for index, row in df_translations.iterrows():
    print(f"Orijinal: {row['original_text']}")
    print(f"Çeviri: {row['translated_text']}\n")
    print("-" * 60)

Çeviri işlemi başladı, lütfen bekleyin...

Orijinal: Foi enviado produto Diferente daquele do anúncio. Solicitei devolver o produto e receber o valor pago, porém ainda não tive resposta. Espero resolver este desagradável problema de forma consensual.
Çeviri: [HATA - ÇEVRİLEMEDİ] Detay: 'coroutine' object has no attribute 'text'

------------------------------------------------------------
Orijinal: Não comprem desta loja! Tentei fazer o cancelamento da compra por 5 vezes e só fui atende depois de ter recusado a entrega do Correio.
Çeviri: [HATA - ÇEVRİLEMEDİ] Detay: 'coroutine' object has no attribute 'text'

------------------------------------------------------------
Orijinal: Excelente
Çeviri: [HATA - ÇEVRİLEMEDİ] Detay: 'coroutine' object has no attribute 'text'

------------------------------------------------------------
Orijinal: O jogo de bolas de bilhar que comprei é totalmente diferente do que foi entregue, sendo diferente, a cor, material e diâmetro da bola. Quanto ao recebi

/tmp/ipykernel_142782/2606541167.py:11: RuntimeWarning: coroutine 'Translator.translate' was never awaited
  translation = translator.translate(text, src='pt', dest='en')


**Insights** 💡
- Bazı kötü review’lar delivery ile ilgili (`wait_time`, kaçırılan teslim tarihi, vb.)
- Ancak bazı kötü review’lar seller veya ürünle ilgili...

Peki iki olası nedeni nasıl ayırt edebiliriz?

Bu Olist’in mutlaka bilmesi gereken bir şey:
- bazı ürünleri katalogdan mı çıkarmalı?
- yoksa bazı seller’ları marketplace’ten mi kaldırmalı?
